In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
import sqlite3
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
llm = ChatGroq(model_name="llama-3.3-70b-versatile",temperature=0.7)

In [3]:
class ChatState(TypedDict):
    messages : Annotated[list[str], add_messages]

In [4]:
def chat_message(state: ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {"messages": [response]}

In [5]:
conn = sqlite3.connect(database="chabot.db", check_same_thread=False)
checkpoint = SqliteSaver(conn)  

In [6]:
graph = StateGraph(ChatState)

graph.add_node("chat_message",chat_message)

graph.add_edge(START, "chat_message")
graph.add_edge("chat_message", END)

workflow = graph.compile(checkpointer=checkpoint)

In [7]:
def get_all_threads():
    all_threads = set()
    for ckpt in checkpoint.list(None):
        all_threads.add(ckpt.config['configurable']['thread_id'])

    return list(all_threads)


In [14]:
thread_id = "1"
config = {"configurable": {"thread_id": thread_id}}
initial_state = {"messages" : [HumanMessage("What are agentic ai")]}
response  = workflow.invoke(initial_state, config=config)

In [15]:
response['messages'][-1].content

'Agentic AI refers to artificial intelligence (AI) systems that are capable of autonomous decision-making, problem-solving, and action-taking, with a sense of agency and self-directed purpose. These systems are designed to operate independently, making decisions and taking actions without human intervention, and are often characterized by their ability to adapt, learn, and evolve over time.\n\nAgentic AI systems typically possess the following characteristics:\n\n1. **Autonomy**: The ability to operate independently, making decisions and taking actions without human intervention.\n2. **Self-directed purpose**: The system has its own goals, objectives, and motivations, which guide its decision-making and behavior.\n3. **Adaptability**: The ability to learn, adapt, and evolve in response to changing circumstances and environments.\n4. **Proactivity**: The ability to anticipate and take proactive measures to achieve its goals and objectives.\n5. **Reactivity**: The ability to respond to c